# SNN Hackathon 2026 – MNE-Python Preprocessing Tutorial
## Day 2: Bad Channels · ICA · Re-referencing · Epochs · Baseline · Rejection · ERP
 
---
 
At this stage you should have already loaded your raw EEG data, merged broken files, resampled and applied band-pass filtering.
Today we finalise the preprocessing pipeline and create clean, baseline-corrected epochs ready for ERP analysis.
 
**Pipeline overview:**
```
Load filtered data (Day 1 output)
         ▼
   1. Bad Channels            (set montage)
         ▼
   2. ICA on continuous data  (fit on 1 Hz HP copy → apply on real data)
         ▼
   3. Re-referencing          (average reference · butterfly plot)
         ▼
   4. Extract Epochs          (time-lock to stimulus onset)
         ▼
   5. Baseline Correction     (−100 to 0 ms)
         ▼
   6. Epoch Rejection         (amplitude threshold ±150 µV)
         ▼
   7. ERP Preview & Save
```
 
> 💡 **Why this order?** Filtering, re-referencing, and averaging are **linear operations** — their order does not matter mathematically.
> However, **ICA should run before re-referencing**: average referencing spreads artifact activity (e.g. eye blinks) across all channels, which can degrade the ICA decomposition.
> Artifact rejection is **nonlinear** (it uses a threshold). Order matters for nonlinear operations — always reject *after* re-referencing and baseline correction.
> *(Luck, S.J. 2014, An Introduction to the Event-Related Potential Technique; erpinfo.org/order-of-steps)*
 

---
## 0 · Imports & Setup

In [ ]:
import os
import os.path as op
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mne.preprocessing import ICA, create_eog_epochs
 
mne.set_log_level('WARNING')  # suppress info messages, keep warnings
%matplotlib inline
 
print('MNE version:', mne.__version__)

## 0.1 · Paths & participant
 
Load the data you processed and saved in tasksheet_1. Also load the original raw data (the .vhdr file) as a separate object - we will need it later for the Independant Component Analysis (ICA).

In [ ]:
##### INSERT YOUR CODE HERE #####

## Part 0.2 Mark Bad Channels
Channels with bad signal quality need to be marked (not removed!) before doing any calculations (like ICA). MNE then automatically excludes these channels from any calculations. 
You can spot channels by creating an interactive plot and scrolling through the EEG recording - however this often lags quite a bit in Jupyter Notebooks - for excercise purposes feel free to just mark any arbitrary channels as bad. 

In [ ]:
##### INSERT YOUR CODE HERE #####

---
## Part 1 · ICA – Artifact Correction on Continuous Data
 
**Independent Component Analysis** decomposes the mixed EEG signal into a set of statistically
**independent source signals** (components). Artifacts like eye blinks have very characteristic
patterns — large, sharp frontal waves — and ICA isolates these into their own components.
We remove those components and reconstruct the signal from the rest.
 
### Why fit ICA on *continuous* data?
 
ICA needs long, uninterrupted stretches of data to reliably find independent sources.
If you epoch first and then fit ICA, artifacts at epoch boundaries may be split across
two trials and missed.

### Why a 1 Hz high-pass copy for *fitting*?
 
ICA is sensitive to slow signal drifts. Our Day 1 data has a 0.2 Hz lower cutoff —
that still allows some drift through, which can confuse ICA.
 
Best practice is:
```
Unfiltered raw  →  copy  →  1 Hz high-pass  →  ica.fit()    ← only for training
ica.apply()         on  filtered raw (0.2 Hz filtered, interpolated)       ← actual output
```
 
### Why ICA *before* re-referencing?
 
Average re-referencing subtracts the mean of all channels from each channel.
This **spreads** artifact activity (e.g. a blink concentrated at Fp1/Fp2) across
*every* electrode, making it harder for ICA to isolate the artifact into a single component.
Running ICA first — on data that still has the original recording reference — gives
the cleanest decomposition.
 
### How to identify artifact components (ICLabel criteria)
 
Use **three complementary plots** — topographic maps, time series, and correlation scores:
 
| Artifact | Topography | Time series | Power spectrum |
|---|---|---|---|
| Eye blink | Large symmetric loading at Fp1/Fp2 | Regular large spikes | Most power <5 Hz |
| Eye movement | Asymmetric frontal (left/right) | Step-function pattern | Most power <5 Hz |
| Heartbeat | Near-linear gradient | Regular QRS complexes (~1/s) | No clear peaks |
| Muscle | Focal, shallow (scalp edge) | Non-stationary bursts | Broadband >20 Hz |
| Brain (KEEP!) | Dipolar map, RV <15% | ERP if epoched | 1/f + peak 5–30 Hz |
 
*(Based on ICLabel taxonomy — labeling.ucsd.edu/tutorial/labels)*
 
> ⚠️  **When in doubt, keep the component.** Removing an relevant component is worse than
> leaving a small artifact residual.

### 📌 Task 1.1 – Prepare the 1 Hz high-pass copy for ICA fitting
 
Create a **copy** of the raw original data and apply a 1 Hz high-pass filter.
This copy is used *only* for `ica.fit()`.

In [ ]:
##### INSERT YOUR CODE HERE #####

### 📌 Task 1.2 – Initialise and fit ICA
 
If you want the same ICA result every time you run the code with the same data, set the 'random_state' (or 'rng', if you are on a newer version of MNE) to some integer value.
Note: a common misconception is, that this will ensure perfect reproducibility across any devices - however that is not the case. Small differences can still occur if hardware, backends etc. differ!

In [ ]:
##### INSERT YOUR CODE HERE #####

### 📌 Task 1.3 – Inspect components visually
 
Use **two plots** to identify artifact components:
 
**1. `plot_components()`** — topographic maps:
- **Eye blinks**: large symmetric loading at Fp1/Fp2
- **Eye movements**: asymmetric frontal pattern
- **Heartbeat**: near-linear gradient across scalp
- **Muscle**: focal, concentrated at scalp edges (shallow source)
- **Brain**: clean dipolar map, typically one polarity on each hemisphere
 
**2. `plot_sources()`** — time series:
- **Eye blinks**: unmistakable large, regular spikes
- **Heartbeat**: rhythmic QRS-shaped pulses (~1 per second)
- **Muscle**: high-frequency irregular bursts
 
> 💡 We plot sources on the data that was NOT 1-Hz filtered so the time scale is meaningful.

In [ ]:
##### INSERT YOUR CODE HERE #####

### 📌 Task 1.4 – Find and exclude artifact components
 
Two approaches — combine for best results:
 
**Option A – Manual:** enter component indices you identified from the plots.
 
**Option B – Automatic (`find_bads_eog()`):** correlates each component with the EOG channel.
High correlation = likely eye artifact. 
- Since our data does not have an EOG Channel, use option 1

In [ ]:
##### INSERT YOUR CODE HERE #####

### 📌 Task 1.5 – Preview: plot_overlay
 
`plot_overlay()` shows a **before/after comparison** at the channel level.
Red = original signal, black = signal after removing the bad components.
 
**Check:** Do the blink spikes disappear? Does the underlying brain signal survive?
The ERP-like shape should remain — only the artifact spikes should disappear.

In [ ]:
##### INSERT YOUR CODE HERE #####

### 📌 Task 1.6 – Apply ICA to the data
 
**Note:** use a .copy() of the data,in case you need the version without the ICA applied later.

In [ ]:
##### INSERT YOUR CODE HERE #####

---
## Part 2 · Re-referencing
 
### What is a reference?
 
An EEG electrode does **not** measure absolute voltage — it always measures the **difference**
between itself and a chosen **reference electrode**. The online-reference is set during recording
(e.g. Cz, one mastoid). 
 
### Average reference
 
The standard solution in ERP research is the **average reference**: re-reference every channel
to the mean of *all* channels simultaneously:
 
```
V_new  =  V_channel  −  mean(all channels)
```
 
We apply this **after ICA** so that artifact activity is not spread across channels before
the decomposition.

### 📌 Task 2.1 – Apply average reference
 
Apply the average reference to the ICA-cleaned data (`raw_clean`).

In [ ]:
##### INSERT YOUR CODE HERE #####

### 📌 Task 3.2 – Butterfly plots: before vs after
 
A **butterfly plot** overlays all channels at once — great for checking signal balance.
 
**What to look for:**
- Are the traces more centred around 0 µV after re-referencing?
- Is the overall amplitude spread smaller?
- Any channel still very different from the rest? → possible bad channel for interpolation

In [ ]:
##### INSERT YOUR CODE HERE #####

---
## Part 3 · Bad Channel Interpolation

### What to interpolate
We now need to interpolate the channels with Bad Signals, which we marked at the start of the script, but now in our filtered, ICA-applied, re-referenced data!
first, check in that data if the channels are already marked as bad - if not look at the start of the script to see which channels you need to mark!

### How does interpolation work?
 
MNE replaces each bad channel with a **weighted average of its spatial neighbours**,
using spherical spline interpolation. For this to work, electrode positions must be set
via a **montage**.
 
```
.interpolate_bads(reset_bads=True)            # replace with neighbour average
```
 
> 💡 `reset_bads=True` clears the bads list after interpolation, so the repaired channel
> is treated as normal in all subsequent steps.

### Plot to see differences
You can use the plot() function from yesterday to take a look at the signal of the interpolated channel before and after!

In [ ]:
##### INSERT YOUR CODE HERE #####

---
## Part 4 · Extract Epochs
 
### What is an epoch?
 
Until now we have been working with **continuous** EEG — one long recording.
For ERP analysis we cut the continuous signal into short, fixed-length segments
called **epochs**, each time-locked to an event marker:
 
```
Continuous:  ─────────────[stim]─────────────[stim]────────────
                               │                   │
                         ┌─────┴──────┐      ┌─────┴──────┐
Epochs:                  │ -100..600ms│      │ -100..600ms│  ...
                         └────────────┘      └────────────┘
```
 
| Parameter | Value | Meaning |
|---|---|---|
| `tmin` | −0.1 s | 100 ms *before* stimulus (pre-stimulus baseline) |
| `tmax` | +0.6 s | 600 ms *after* stimulus |
| `baseline` | `None` | We apply this separately in Part 5 |
| `preload` | `True` | Load all epochs into RAM immediately |
| `reject` | `None` | Applied separately in Part 6 |

### 📌 Task 4.1 – Find events
 
Event markers were recorded as annotations. Convert them to an array MNE can use for epoching.

In [ ]:
##### INSERT YOUR CODE HERE #####

### 📌 Task 4.2 – Create epochs
 
Fill in the correct event codes from the dictionary you printed above. Data was recorded while participants did an Go/NoGo Task; relevant codes are codes 1 through 4. Triggers 1, 2, 3 = Go stimuli; Trigger 4 = NoGo stimulus.

In [ ]:
##### INSERT YOUR CODE HERE #####

### 📌 Task 4.3 – (Bonus) Epoch image plot
 
An **epoch image plot** stacks all individual epochs as rows (colour = amplitude).
Noisy trials stand out as bright/dark rows, and consistent ERP patterns show up
as horizontal bands of colour.

In [ ]:
##### INSERT YOUR CODE HERE #####

---
## Part 5 · Baseline Correction
 
### What and why?
 
EEG signals drift — the overall voltage level shifts gradually over time.
This means the starting voltage of each epoch is slightly different,
making cross-trial comparisons unfair.
 
**Baseline correction** removes this drift:

> 💡 Baseline correction is a **linear operation** — its order relative to averaging
> doesn't matter. But always apply it *before* artifact rejection, so rejection
> decisions are made on corrected data. *(Luck, 2014; erpinfo.org)*

### 📌 Task 5.1 – Apply baseline correction
 

In [ ]:
##### INSERT YOUR CODE HERE #####

### 📌 Task 5.2 – Verify the baseline correction
 
`.get_data()` returns data in **Volts**. Multiply by `1e6` to convert to µV.

In [ ]:
##### INSERT YOUR CODE HERE #####

---
## Part 6 · Epoch Rejection
 
### What and why?
 
Even after ICA, some epochs may still contain large residual artifacts:
- Muscle bursts (participant coughed or moved their head)
- Electrode pops (sudden voltage jump in one channel)
- Any artifact ICA did not decompose into its own component
 
We use **threshold rejection**: discard any epoch where *any* EEG channel exceeds ±150 µV.
 
> 💡 Thresholds in MNE are always in **Volts**, not µV. So 150 µV = `150e-6` V.
 
> ⚠️ Artifact rejection is **nonlinear** — always apply it *after* re-referencing and
> baseline correction, never before. *(Luck, 2014; erpinfo.org/order-of-steps)*
 
> 💡 If one channel causes most rejections, consider **interpolating** that channel
> (replacing it with a weighted average of its neighbours) before rejecting epochs.

### 📌 Task 6.1 – Reject bad epochs

In [ ]:
##### INSERT YOUR CODE HERE #####

### 📌 Task 6.2 – Plot the drop log
 
`plot_drop_log()` shows *which* epochs were dropped and *which channel* caused each rejection.
 
If one channel appears very often → consider interpolation instead of rejection.

In [ ]:
##### INSERT YOUR CODE HERE #####

---
## Part 7 · Save
 
Preprocessing is done. We save the clean epochs as a `.fif` file.
This is the output that feeds into Day 3 — where we will compute the actual ERPs,
compare Go vs. NoGo, and look at N2 and P3 components.
 
> 💡 Epoch files must end with `-epo.fif` or `_epo.fif` (MNE naming convention).

### 📌 Task 7.1 – Print summary

In [ ]:
##### INSERT YOUR CODE HERE #####

  Summary for B04
  Clean epochs total:     192
  Conditions:             ['Go/1', 'Go/2', 'Go/3', 'NoGo']
  Time window:            -102 to 602 ms
  Sampling rate:          256.0 Hz
  ICA components removed: [0]
  Interpolated channels:  P10
  Go/1  : 46 epochs
  Go/2  : 49 epochs
  Go/3  : 46 epochs
  NoGo  : 51 epochs


### 📌 Task 7.2 – Save the clean epochs

In [ ]:
##### INSERT YOUR CODE HERE #####

---
**Good luck!**

*References: MNE ERP tutorial (mne.tools/stable/auto_tutorials/evoked/30_eeg_erp.html) · Luck (2014) via erpinfo.org/order-of-steps · ICLabel taxonomy via labeling.ucsd.edu/tutorial/labels*